<a href="https://colab.research.google.com/github/DomeRomeroC/audioDispensador/blob/main/colab/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
    <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://docs.nerf.studio/_images/logo-dark.png">
    <source media="(prefers-color-scheme: light)" srcset="https://docs.nerf.studio/_images/logo.png">
    <img alt="nerfstudio" src="https://docs.nerf.studio/_images/logo.png" width="400">
    </picture>
</p>


# Nerfstudio: A collaboration friendly studio for NeRFs


![GitHub stars](https://img.shields.io/github/stars/nerfstudio-project/nerfstudio?color=gold&style=social)

This colab shows how to train and view NeRFs from Nerfstudio both on pre-made datasets or from your own videos/images.

\\

Credit to [NeX](https://nex-mpi.github.io/) for Google Colab format.

In [1]:
!nvidia-smi

Sat Jul  4 00:22:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Frequently Asked Questions

*  **Downloading custom data is stalling (no output):**
    * This is a bug in Colab. The data is processing, but may take a while to complete. You will know processing completed if `data/nerfstudio/custom_data/transforms.json` exists. Terminating the cell early will result in not being able to train.
*  **Processing custom data is taking a long time:**
    * The time it takes to process data depends on the number of images and its resolution. If processing is taking too long, try lowering the resolution of your custom data.
*  **Error: Data processing did not complete:**
    * This means that the data processing script did not fully complete. This could be because there were not enough images, or that the images were of low quality. We recommend images with little to no motion blur and lots of visual overlap of the scene to increase the chances of successful processing.
*   **Training is not showing progress**:
    * The lack of output is a bug in Colab. You can see the training progress from the viewer.
* **Viewer Quality is bad / Low resolution**:
    * This may be because more GPU is being used on training that rendering the viewer. Try pausing training or decreasing training utilization.
* **WARNING: Running pip as the 'root' user...:**:
    * This and other pip warnings or errors can be safely ignored.
* **Other problems?**
    * Feel free to create an issue on our [GitHub repo](https://github.com/nerfstudio-project/nerfstudio).


In [2]:
# @markdown <h1>Install Nerfstudio and Dependencies (~8 min)</h1>

%cd /content/
!pip install --upgrade pip
!pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --extra-index-url https://download.pytorch.org/whl/cu118

# Installing TinyCuda
%cd /content/
!gdown "https://drive.google.com/u/1/uc?id=1-7x7qQfB7bIw2zV4Lr6-yhvMpjXC84Q5&confirm=t"
!pip install tinycudann-1.7-cp310-cp310-linux_x86_64.whl

# Installing COLMAP
%cd /content/
!apt-get install colmap

# Install nerfstudio
%cd /content/
!pip install git+https://github.com/nerfstudio-project/nerfstudio.git

/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement torch==2.0.1+cu118 (from versions: 2.2.0, 2.2.0+cu118, 2.2.1, 2.2.1+cu118, 2.2.2, 2.2.2+cu118, 2.3.0, 2.3.0+cu118, 2.3.1, 2.3.1+cu118, 2.4.0, 2.4.0+cu118, 2.4.1, 2.4.1+cu118, 2.5.0, 2.5.0+cu118, 2.5.1, 2.5.1+cu118, 2.6.0, 2.6.0+cu118, 2.7.0, 2.7.0+cu118, 2.7.1, 2.7.1+cu118, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1)
ERROR: No matching distribution found for torch==2.0.1+cu118
/content
Downloading...
From: https://drive.google.com/u/1/uc?id=1-7x7qQfB7bIw2zV4Lr6-yhvMpjXC84Q5&confirm=t
To: /content/tinycudann-1.7-cp310-cp310-linux_x86_64.whl
100% 31.2M/31.2M [00:00<00:00, 36.6MB/s]
ERROR: tinycudann-1.7-cp310-

In [11]:
# @markdown <h1> Downloading and Processing Data</h1>
# @markdown <h3>Pick the preset scene or upload your own images/video</h3>
import glob
import os

from google.colab import files
from IPython.core.display import HTML, display

scene = "🎥 upload your own video"  # @param ['🖼 poster', '🚜 dozer', '🌄 desolation', '📤 upload your images' , '🎥 upload your own video', '🔺 upload Polycam data', '💽 upload your own Record3D data']
scene = " ".join(scene.split(" ")[1:])

if scene == "upload Polycam data":
    %cd /content/
    !mkdir -p /content/data/nerfstudio/custom_data
    %cd /content/data/nerfstudio/custom_data/
    uploaded = files.upload()
    dir = os.getcwd()
    if len(uploaded.keys()) > 1:
        print("ERROR, upload a single .zip file when processing Polycam data")
    dataset_dir = [os.path.join(dir, f) for f in uploaded.keys()][0]
    !ns-process-data polycam --data $dataset_dir --output-dir /content/data/nerfstudio/custom_data/
    scene = "custom_data"
elif scene == "upload your own Record3D data":
    display(HTML("<h3>Zip your Record3D folder, and upload.</h3>"))
    display(
        HTML(
            '<h3>More information on Record3D can be found <a href="https://docs.nerf.studio/en/latest/quickstart/custom_dataset.html#record3d-capture" target="_blank">here</a>.</h3>'
        )
    )
    %cd /content/
    !mkdir -p /content/data/nerfstudio/custom_data
    %cd /content/data/nerfstudio/custom_data/
    uploaded = files.upload()
    dir = os.getcwd()
    preupload_datasets = [os.path.join(dir, f) for f in uploaded.keys()]
    record_3d_zipfile = preupload_datasets[0]
    !unzip $record_3d_zipfile -d /content/data/nerfstudio/custom_data
    custom_data_directory = glob.glob("/content/data/nerfstudio/custom_data/*")[0]
    !ns-process-data record3d --data $custom_data_directory --output-dir /content/data/nerfstudio/custom_data/
    scene = "custom_data"
elif scene in ["upload your images", "upload your own video"]:
    display(HTML("<h3>Select your custom data</h3>"))
    display(HTML("<p/>You can select multiple images by pressing ctrl, cmd or shift and click.<p>"))
    display(
        HTML(
            "<p/>Note: This may take time, especially on higher resolution inputs, so we recommend to download dataset after creation.<p>"
        )
    )
    !mkdir -p /content/data/nerfstudio/custom_data
    if scene == "upload your images":
        !mkdir -p /content/data/nerfstudio/custom_data/raw_images
        %cd /content/data/nerfstudio/custom_data/raw_images
        uploaded = files.upload()
        dir = os.getcwd()
    else:
        %cd /content/data/nerfstudio/custom_data/
        uploaded = files.upload()
        dir = os.getcwd()
    preupload_datasets = [os.path.join(dir, f) for f in uploaded.keys()]
    del uploaded
    %cd /content/

    if scene == "upload your images":
        !ns-process-data images --data /content/data/nerfstudio/custom_data/raw_images --output-dir /content/data/nerfstudio/custom_data/
    else:
        video_path = preupload_datasets[0]
        !ns-process-data video --data $video_path --output-dir /content/data/nerfstudio/custom_data/

    scene = "custom_data"
else:
    %cd /content/
    !ns-download-data nerfstudio --capture-name=$scene

print("Data Processing Succeeded!")

/content/data/nerfstudio/custom_data


Saving VIDEO.mp4 to VIDEO.mp4
/content
Number of frames in video: 1491
Extracting 373 frames in evenly spaced intervals
[01:14:25] 🎉 Done converting video to images.                                                 ]8;id=597324;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=359010;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py#227\227]8;;\
(   ●  ) Converting video to images...
──────────────────────────────────────────────  💀 💀 💀 ERROR 💀 💀 💀  ───────────────────────────────────────────────
Error running command: colmap feature_extractor --database_path /content/data/nerfstudio/custom_data/colmap/database.db 
--image_path /content/data/nerfstudio/custom_data/images --ImageReader.single_camera 1 --ImageReader.camera_model OPENCV
--SiftExtraction.use_gpu 1
───────────────────────────────────────────────────────────────────────────────────────────────────────

In [12]:
!ls /content/data/nerfstudio/custom_data

 colmap   images_2   images_8	'WhatsApp Video 2026-07-03 at 7.57.08 PM.mp4'
 images   images_4   VIDEO.mp4


In [14]:
!find /content/data/nerfstudio/custom_data -name "transforms.json"

In [15]:
!rm -rf /content/data/nerfstudio/custom_data

!ns-process-data video \
  --data /content/video.mp4 \
  --output-dir /content/data/nerfstudio/custom_data \
  --num-frames-target 100

Error: Video does not exist: /content/video.mp4


In [17]:
!find /content -name "*.mp4" -o -name "*.MP4"

In [18]:
!rm -rf /content/data/nerfstudio/custom_data

!ns-process-data video \
  --data /content/VIDEO.mp4 \
  --output-dir /content/data/nerfstudio/custom_data \
  --num-frames-target 100

Error: Video does not exist: /content/VIDEO.mp4


In [19]:
from google.colab import files
uploaded = files.upload()

Saving VIDEO.mp4 to VIDEO.mp4


In [22]:
import shutil

nombre = list(uploaded.keys())[0]
shutil.move(nombre, "/content/video.mp4")

print("Video guardado como /content/video.mp4")

FileNotFoundError: [Errno 2] No such file or directory: 'VIDEO.mp4'

In [24]:
from google.colab import files
import os, shutil

# Ir a la carpeta principal de Colab
os.chdir("/content")

# Subir el video otra vez
uploaded = files.upload()

# Tomar el nombre real del archivo subido
nombre = list(uploaded.keys())[0]
print("Archivo subido:", nombre)

# Copiarlo con un nombre simple
shutil.copyfile(nombre, "/content/video.mp4")

print("Video guardado como /content/video.mp4")
print("Existe:", os.path.exists("/content/video.mp4"))

Saving VIDEO.mp4 to VIDEO (1).mp4
Archivo subido: VIDEO (1).mp4
Video guardado como /content/video.mp4
Existe: True


In [25]:
!rm -rf /content/data/nerfstudio/custom_data

!ns-process-data video \
  --data /content/video.mp4 \
  --output-dir /content/data/nerfstudio/custom_data \
  --num-frames-target 100

Number of frames in video: 1491
Extracting 107 frames in evenly spaced intervals
[01:46:13] 🎉 Done converting video to images.                                                 ]8;id=313368;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=292498;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py#227\227]8;;\
(  ●   ) Converting video to images...
──────────────────────────────────────────────  💀 💀 💀 ERROR 💀 💀 💀  ───────────────────────────────────────────────
Error running command: colmap feature_extractor --database_path /content/data/nerfstudio/custom_data/colmap/database.db 
--image_path /content/data/nerfstudio/custom_data/images --ImageReader.single_camera 1 --ImageReader.camera_model OPENCV
--SiftExtraction.use_gpu 1
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
qt.qpa.xcb: could not

In [26]:
!ls /content/data/nerfstudio/custom_data

colmap	images	images_2  images_4  images_8


In [27]:
!rm -rf /content/data/nerfstudio/custom_data

!ns-process-data video \
  --data /content/video.mp4 \
  --output-dir /content/data/nerfstudio/custom_data \
  --num-frames-target 50

Number of frames in video: 1491
Extracting 52 frames in evenly spaced intervals
[01:52:19] 🎉 Done converting video to images.                                                 ]8;id=176167;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=162406;file:///usr/local/lib/python3.12/dist-packages/nerfstudio/process_data/process_data_utils.py#227\227]8;;\
(     ●) Converting video to images...
──────────────────────────────────────────────  💀 💀 💀 ERROR 💀 💀 💀  ───────────────────────────────────────────────
Error running command: colmap feature_extractor --database_path /content/data/nerfstudio/custom_data/colmap/database.db 
--image_path /content/data/nerfstudio/custom_data/images --ImageReader.single_camera 1 --ImageReader.camera_model OPENCV
--SiftExtraction.use_gpu 1
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
qt.qpa.xcb: could not 

In [28]:
!ls /content/data/nerfstudio/custom_data

colmap	images	images_2  images_4  images_8


In [29]:
import os, json, math
import numpy as np
from PIL import Image, ImageDraw

out_dir = "/content/data/test_20"
img_dir = os.path.join(out_dir, "images")
os.makedirs(img_dir, exist_ok=True)

W, H = 512, 512
fx = fy = 480.0
cx, cy = W / 2, H / 2

# Crear una nube de puntos 3D simple: cubo + piso
points = []
colors = []

# Cubo de puntos con colores
for x in np.linspace(-0.6, 0.6, 18):
    for y in np.linspace(-0.6, 0.6, 18):
        for z in np.linspace(-0.6, 0.6, 18):
            if abs(x) > 0.52 or abs(y) > 0.52 or abs(z) > 0.52:
                points.append([x, y, z])
                colors.append([
                    int((x + 0.6) / 1.2 * 255),
                    int((y + 0.6) / 1.2 * 255),
                    int((z + 0.6) / 1.2 * 255),
                ])

# Piso con puntos grises
for x in np.linspace(-1.2, 1.2, 30):
    for y in np.linspace(-1.2, 1.2, 30):
        points.append([x, y, -0.75])
        colors.append([180, 180, 180])

points = np.array(points, dtype=float)
colors = np.array(colors, dtype=np.uint8)

def look_at(camera_pos, target=np.array([0, 0, 0]), up=np.array([0, 0, 1])):
    camera_pos = np.array(camera_pos, dtype=float)
    forward = target - camera_pos
    forward = forward / np.linalg.norm(forward)

    right = np.cross(forward, up)
    right = right / np.linalg.norm(right)

    true_up = np.cross(right, forward)
    true_up = true_up / np.linalg.norm(true_up)

    c2w = np.eye(4)
    c2w[:3, 0] = right
    c2w[:3, 1] = true_up
    c2w[:3, 2] = -forward
    c2w[:3, 3] = camera_pos
    return c2w

frames = []

for i in range(20):
    angle = 2 * math.pi * i / 20
    cam_pos = np.array([
        2.6 * math.cos(angle),
        2.6 * math.sin(angle),
        1.0
    ])

    c2w = look_at(cam_pos)
    w2c = np.linalg.inv(c2w)

    img = Image.new("RGB", (W, H), (245, 245, 245))
    draw = ImageDraw.Draw(img)

    # Convertir puntos mundo -> cámara
    pts_h = np.concatenate([points, np.ones((len(points), 1))], axis=1)
    cam = (w2c @ pts_h.T).T[:, :3]

    # En convención OpenGL, los puntos delante de cámara tienen z negativo
    z = cam[:, 2]
    valid = z < -0.05

    cam_valid = cam[valid]
    col_valid = colors[valid]
    depth = -cam_valid[:, 2]

    u = fx * (cam_valid[:, 0] / depth) + cx
    v = fy * (-cam_valid[:, 1] / depth) + cy

    order = np.argsort(depth)[::-1]  # dibujar de lejos a cerca

    for idx in order:
        uu, vv = u[idx], v[idx]
        if 0 <= uu < W and 0 <= vv < H:
            r = max(1, int(5 / depth[idx]))
            color = tuple(int(c) for c in col_valid[idx])
            draw.ellipse((uu-r, vv-r, uu+r, vv+r), fill=color)

    filename = f"frame_{i:03d}.png"
    img.save(os.path.join(img_dir, filename))

    frames.append({
        "file_path": f"images/{filename}",
        "transform_matrix": c2w.tolist()
    })

transforms = {
    "camera_model": "OPENCV",
    "fl_x": fx,
    "fl_y": fy,
    "cx": cx,
    "cy": cy,
    "w": W,
    "h": H,
    "k1": 0.0,
    "k2": 0.0,
    "p1": 0.0,
    "p2": 0.0,
    "frames": frames
}

with open(os.path.join(out_dir, "transforms.json"), "w") as f:
    json.dump(transforms, f, indent=2)

print("Dataset de prueba creado en:", out_dir)
print("Imágenes:", len(frames))
print("Archivo transforms.json creado:", os.path.exists(os.path.join(out_dir, "transforms.json")))

Dataset de prueba creado en: /content/data/test_20
Imágenes: 20
Archivo transforms.json creado: True


In [30]:
!ls /content/data/test_20
!ls /content/data/test_20/images | head

images	transforms.json
frame_000.png
frame_001.png
frame_002.png
frame_003.png
frame_004.png
frame_005.png
frame_006.png
frame_007.png
frame_008.png
frame_009.png


In [ ]:
!ns-train nerfacto \
  --data /content/data/test_20 \
  --max-num-iterations 1000 \
  --viewer.websocket-port 7007 \
  --viewer.make-share-url True

2026-07-04 01:59:30.175317: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
[01:59:43] Using --data alias for --data.pipeline.datamanager.data                                          ]8;id=834153;file:///usr/local/lib/python3.12/dist-packages/nerf

In [2]:
!ls -R /content/data/nerfstudio


/content/data/nerfstudio:
poster

/content/data/nerfstudio/poster:


In [3]:
!rm -rf /content/data/nerfstudio/poster

In [4]:
!python -c "import nerfstudio; print('Nerfstudio instalado correctamente')"

Nerfstudio instalado correctamente


In [5]:
!rm -rf /content/data/nerfstudio/poster

In [6]:
!ns-download-data nerfstudio --capture-name=dozer --save-dir=/content/data/nerfstudio

2026-07-04 00:40:53.124756: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 273, in download
    url = get_url_from_gdrive_confirmation(res.text)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 55, in get_url_from_gdrive_confirmation
    raise FileURLRetrievalError(
gdown.exceptions.FileURLRetrievalError: Cannot retrieve the public link of the file. You may need to change the permission to 'Anyone with the link', or have had many accesses. Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

During handling of the above exception, another except

In [8]:
!rm -rf /content/data/nerfstudio
!rm -rf /content/data/blender

In [9]:
!ns-download-data blender --save-dir /content/data

2026-07-04 00:42:50.470525: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 273, in download
    url = get_url_from_gdrive_confirmation(res.text)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 55, in get_url_from_gdrive_confirmation
    raise FileURLRetrievalError(
gdown.exceptions.FileURLRetrievalError: Cannot retrieve the public link of the file. You may need to change the permission to 'Anyone with the link', or have had many accesses. Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

During handling of the above exception, another except

In [ ]:
# @markdown <h1>Start Training</h1>

%cd /content
!pip install colab-xterm
%load_ext colabxterm
%env TERM=xterm
from IPython.display import clear_output

clear_output(wait=True)
if os.path.exists(f"data/nerfstudio/{scene}/transforms.json"):
    print(
        "\033[1m"
        + "Copy and paste the following command into the terminal window that pops up under this cell."
        + "\033[0m"
    )
    print(
        f"ns-train nerfacto --viewer.websocket-port 7007 --viewer.make-share-url True nerfstudio-data --data data/nerfstudio/{scene} --downscale-factor 4"
    )
    print()
    %xterm
else:
    from IPython.core.display import HTML, display

    display(HTML('<h3 style="color:red">Error: Data processing did not complete</h3>'))
    display(HTML("<h3>Please re-run `Downloading and Processing Data`, or view the FAQ for more info.</h3>"))

In [ ]:
# @title # Render Video { vertical-output: true }
# @markdown <h3>Export the camera path from within the viewer, then run this cell.</h3>
# @markdown <h5>The rendered video should be at renders/output.mp4!</h5>


base_dir = "/content/outputs/unnamed/nerfacto/"
training_run_dir = base_dir + os.listdir(base_dir)[0]

from IPython.core.display import HTML, display

display(HTML("<h3>Upload the camera path JSON.</h3>"))
%cd $training_run_dir
uploaded = files.upload()
uploaded_camera_path_filename = list(uploaded.keys())[0]

config_filename = training_run_dir + "/config.yml"
camera_path_filename = training_run_dir + "/" + uploaded_camera_path_filename
camera_path_filename = camera_path_filename.replace(" ", "\\ ").replace("(", "\\(").replace(")", "\\)")

%cd /content/
!ns-render camera-path --load-config $config_filename --camera-path-filename $camera_path_filename --output-path renders/output.mp4

[19:48:48] Skipping 0 files in dataset split train.                                          ]8;id=527413;file:///content/nerfstudio/nerfstudio/data/dataparsers/nerfstudio_dataparser.py\nerfstudio_dataparser.py]8;;\:]8;id=243595;file:///content/nerfstudio/nerfstudio/data/dataparsers/nerfstudio_dataparser.py#91\91]8;;\
           Skipping 0 files in dataset split test.                                           ]8;id=109270;file:///content/nerfstudio/nerfstudio/data/dataparsers/nerfstudio_dataparser.py\nerfstudio_dataparser.py]8;;\:]8;id=464675;file:///content/nerfstudio/nerfstudio/data/dataparsers/nerfstudio_dataparser.py#91\91]8;;\
Loading data batch ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
/usr/local/lib/python3.7/site-packages/torch/utils/data/dataloader.py:566: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Plea